# TRABAJO FINAL : Evaluación del Desempeño Sísmico y Predicción del Índice de Daño Estructural en Edificaciones mediante Machine Learning Supervisado
## Objetivo
Implementar un modelo de Machine Learning para predecir el índice de daño estructural usando dataset de sismos de Kaggle.
  
Path: /kaggle/input/datasets/ziya07/pre-earthquake-prediction-dataset/seismic_data.csv

## Paso 1:  CARGA DE DATOS
1.1 Carga del dataset `seismic_data.csv` y exploración inicial.

In [ ]:
import pandas as pd
import numpy as np

# Cargar el dataset
df = pd.read_csv('seismic_data.csv')

# Mostrar información básica
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nTipos de datos:")
print(df.dtypes)

# Primeras 5 filas
display(df.head())

1.2 Descripción y estadística de Datos

In [ ]:
# Información general
print("=== Info del Dataset ===")
df.info()

In [ ]:
# Estadísticas descriptivas
print("=== Estadísticas Descriptivas ===")
display(df.describe(include='all').T)

In [ ]:
# Verificar valores nulos
print("=== Valores Nulos ===")
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
null_df = pd.DataFrame({'Valores_Nulos': null_counts, 'Porcentaje': null_percent})
display(null_df[null_df['Valores_Nulos'] > 0].sort_values('Valores_Nulos', ascending=False))

## Paso 2:  Análisis Exploratorio de datos

### 2.1 Visualización de Distribución de Características (Features)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Variables numéricas (features)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target_vars = [
    'Predicted Max Inter-Story Drift Ratio (%)',
    'Predicted Max Roof Displacement (m)',
    'Predicted Base Shear Force (kN)',
    'Predicted Structural Acceleration (m/s²)',
    'Predicted Damage Index (0–1 Scale)',
    'Predicted Collapse Probability (%)'
]
target_vars = [t for t in target_vars if t in df.columns]
feature_cols = [c for c in numeric_cols if c not in target_vars]

print(f"Features numéricas: {len(feature_cols)}")
print(f"Targets: {len(target_vars)}")

In [ ]:
# Histogramas + KDE para features
n_cols = 4
n_rows = int(np.ceil(len(feature_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(feature_cols):
    ax = axes[idx]
    sns.histplot(data=df, x=col, kde=True, bins=30, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(f'Distribución: {col}', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

for idx in range(len(feature_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para features (detección de outliers)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(feature_cols):
    ax = axes[idx]
    sns.boxplot(data=df, y=col, ax=ax, color='lightcoral')
    ax.set_title(f'Boxplot: {col}', fontsize=10)
    ax.set_ylabel('')

for idx in range(len(feature_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

### 2.2 Visualización de Variables Objetivo (Targets)

In [ ]:
# Estadísticas de targets
print("=== Estadísticas de Targets ===")
display(df[target_vars].describe().T)

In [ ]:
# Distribución de targets - Histogramas + KDE con media/mediana
n_targets = len(target_vars)
n_cols = 3
n_rows = int(np.ceil(n_targets / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

for idx, target in enumerate(target_vars):
    ax = axes[idx]
    sns.histplot(data=df, x=target, kde=True, bins=40, ax=ax, color='darkorange', alpha=0.7, stat='density')
    mean_val = df[target].mean()
    median_val = df[target].median()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, label=f'Media: {mean_val:.3f}')
    ax.axvline(median_val, color='green', linestyle='--', linewidth=1.5, label=f'Mediana: {median_val:.3f}')
    ax.set_title(f'Target: {target}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45)

for idx in range(n_targets, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots de targets por zona sísmica
if 'Seismic Zone' in df.columns:
    zone_order = ['Low', 'High', 'Very High']
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten()
    
    for idx, target in enumerate(target_vars):
        ax = axes[idx]
        sns.boxplot(data=df, x='Seismic Zone', y=target, ax=ax, order=zone_order, palette='Set2')
        ax.set_title(f'{target} por Zona Sísmica', fontsize=11, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel('')
    
    for idx in range(n_targets, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Violin plots de targets por zona sísmica
if 'Seismic Zone' in df.columns:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten()
    
    for idx, target in enumerate(target_vars):
        ax = axes[idx]
        sns.violinplot(data=df, x='Seismic Zone', y=target, ax=ax, order=zone_order, palette='Set2', inner='quartile')
        ax.set_title(f'{target} por Zona Sísmica (Violin)', fontsize=11, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.set_xlabel('')
    
    for idx in range(n_targets, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

### 2.3 Correlación Features vs Targets

In [ ]:
# Heatmap de correlación: Features vs Targets
corr_matrix = df[numeric_cols].corr()
corr_ft = corr_matrix.loc[feature_cols, target_vars]

plt.figure(figsize=(14, 16))
sns.heatmap(corr_ft, 
            annot=True, 
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlación'},
            annot_kws={'size': 8})
plt.title('Correlación: Features (filas) vs Targets (columnas)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 features más correlacionadas con cada target
print("=== Top 10 Features por Correlación con cada Target ===")
for target in target_vars:
    target_corr = corr_ft[target].abs().sort_values(ascending=False).head(10)
    print(f"\n📌 {target}")
    display(target_corr.to_frame('|Correlación|').style.format({'|Correlación|': '{:.4f}'}).background_gradient(cmap='Reds'))

### 2.4 Matriz de Correlación entre Variables Numéricas
Análisis de la matriz de correlación completa entre todas las variables numéricas del dataset (features y targets) para identificar relaciones lineales, multicolinealidad y patrones de asociación.

In [ ]:
# Matriz de correlación completa entre todas las variables numéricas
corr_matrix_full = df[numeric_cols].corr(method='pearson')

plt.figure(figsize=(20, 16))
sns.heatmap(corr_matrix_full,
            annot=True,
            fmt='.2f',
            cmap='RdBu_r',
            center=0,
            linewidths=0.5,
            linecolor='gray',
            cbar_kws={'shrink': 0.7, 'label': 'Coeficiente de Correlación de Pearson'},
            annot_kws={'size': 7})
plt.title('Matriz de Correlación - Variables Numéricas', fontsize=16, pad=20, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de correlación con estilo visual
styled_corr = corr_matrix_full.style.background_gradient(cmap='RdBu_r', axis=None, center=0).format('{:.2f}')
styled_corr

In [ ]:
# Detectar pares con alta correlación para identificar multicolinealidad
high_corr_pairs = []
for i in range(len(corr_matrix_full.columns)):
    for j in range(i + 1, len(corr_matrix_full.columns)):
        corr_val = corr_matrix_full.iloc[i, j]
        if abs(corr_val) > 0.8:
            high_corr_pairs.append({
                'Variable 1': corr_matrix_full.columns[i],
                'Variable 2': corr_matrix_full.columns[j],
                'Correlación': round(corr_val, 4)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlación', ascending=False)
print("=== Pares de Variables con Correlación Alta (|r| > 0.8) ===")
if len(high_corr_df) > 0:
    display(high_corr_df)
else:
    print("No se encontraron pares con correlación > 0.8")
print(f"\nTotal de pares con alta correlación: {len(high_corr_df)}")

In [ ]:
# Correlación de cada feature con cada target
print("=== Correlación de cada Feature con cada Target ===")
corr_features_targets = corr_matrix_full.loc[feature_cols, target_vars]

fig, ax = plt.subplots(figsize=(14, max(8, len(feature_cols) * 0.4)))
sns.heatmap(corr_features_targets,
            annot=True,
            fmt='.3f',
            cmap='RdBu_r',
            center=0,
            linewidths=0.5,
            cbar_kws={'shrink': 0.7, 'label': 'Correlación'},
            annot_kws={'size': 8})
ax.set_title('Correlación Features vs Targets', fontsize=13, pad=15, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.5 Pairplot: Targets vs Top Features

In [ ]:
# Pairplot: Damage Index vs sus top 4 features correlacionadas
main_target = 'Predicted Damage Index (0–1 Scale)'
if main_target in target_vars:
    top_features = corr_ft[main_target].abs().sort_values(ascending=False).head(4).index.tolist()
    pair_vars = top_features + [main_target]
    
    print(f"Variables para pairplot: {pair_vars}")
    
    sample_df = df[pair_vars].sample(n=min(800, len(df)), random_state=42)
    
    g = sns.pairplot(sample_df, diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20}, height=2.5)
    g.fig.suptitle(f'Pairplot: {main_target} vs Top 4 Features Correlacionadas', y=1.02, fontsize=14)
    plt.show()

### 2.6 Distribución por Variables Categóricas

In [ ]:
# Distribución de variables categóricas
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    plt.figure(figsize=(10, 4))
    
    # Countplot
    plt.subplot(1, 2, 1)
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, palette='viridis')
    plt.title(f'Frecuencia: {col}', fontweight='bold')
    plt.xticks(rotation=45)
    
    # Proporción
    plt.subplot(1, 2, 2)
    prop = df[col].value_counts(normalize=True) * 100
    prop.plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('viridis', len(prop)))
    plt.title(f'Proporción: {col}', fontweight='bold')
    plt.ylabel('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Targets vs variables categóricas (boxplots)
if cat_cols and target_vars:
    for cat in cat_cols:
        n_unique = df[cat].nunique()
        if n_unique <= 6:
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
            axes = axes.flatten()
            
            for idx, target in enumerate(target_vars):
                ax = axes[idx]
                sns.boxplot(data=df, x=cat, y=target, ax=ax, palette='Set3')
                ax.set_title(f'{target} por {cat}', fontsize=10)
                ax.tick_params(axis='x', rotation=45)
                ax.set_xlabel('')
            
            for idx in range(n_targets, len(axes)):
                axes[idx].set_visible(False)
            
            plt.suptitle(f'Targets por {cat}', fontsize=14, y=1.02)
            plt.tight_layout()
            plt.show()

## Paso 3:  Preprocesamiento de Datos

### 3.1 Codificación de Variables Categóricas y Escalado de Características Numéricas
Conversión de variables categóricas mediante One-Hot Encoding y normalización de características numéricas usando StandardScaler, para preparar los datos para modelos de Machine Learning.

In [ ]:
# Importar herramientas de preprocesamiento
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Definir columnas categóricas y numéricas
cat_cols_encoded = df.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols_scaled = numeric_cols.copy()

print(f"Variables categóricas: {len(cat_cols_encoded)}")
print(f"Variables numéricas: {len(num_cols_scaled)}")
print(f"\nCategóricas: {cat_cols_encoded}")
print(f"Numéricas: {num_cols_scaled}")

In [ ]:
# One-Hot Encoding de variables categóricas
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore', dtype=int)
encoded_cols = encoder.fit_transform(df[cat_cols_encoded])
encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(cat_cols_encoded))

print(f"Shape original (features): {df[feature_cols].shape}")
print(f"Shape codificado (categorical): {encoded_df.shape}")
print(f"\nColumnas codificadas:")
display(encoded_df.head())

In [ ]:
# Concatenar features numéricas (sin codificar) con variables categóricas codificadas
df_encoded = pd.concat([df[feature_cols].reset_index(drop=True), encoded_df], axis=1)

print(f"Dataset codificado - Features: {df_encoded.shape}")
display(df_encoded.head())

In [ ]:
# Escalado de características numéricas con StandardScaler (media=0, std=1)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[feature_cols])
df_scaled = pd.DataFrame(scaled_features, columns=feature_cols)

print("=== Estadísticas antes del escalado ===")
display(df[feature_cols].describe().T[["mean", "std"]])
print("\n=== Estadísticas después del escalado ===")
display(df_scaled.describe().T[["mean", "std"]])

In [ ]:
# Dataset final: features numéricas escaladas + categóricas codificadas
df_final = pd.concat([df_scaled, encoded_df], axis=1)

print("=== Dataset Final Preparado ===")
print(f"Total de columnas (features): {df_final.shape[1]}")
print(f"Total de filas: {df_final.shape[0]}")
display(df_final.head())

In [ ]:
# Pipeline de preprocesamiento completo usando ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore', dtype=int), cat_cols_encoded)
    ],
    remainder='drop'
)

# Ajustar el preprocessor al dataset
X_processed = preprocessor.fit_transform(df)
processed_df = pd.DataFrame(X_processed, columns=preprocessor.get_feature_names_out())

print(f"Shape del dataset procesado por Pipeline: {processed_df.shape}")
display(processed_df.head())

### 3.2 División de Datos en Conjuntos de Entrenamiento y Prueba (80/20)
Separación de los datos en conjuntos de entrenamiento (80%) y prueba (20%) para evaluar el rendimiento del modelo y evitar sobreajuste.

In [ ]:
# Importar función de división
from sklearn.model_selection import train_test_split

# Definir variables predictoras (X) y target principal (y)
main_target = 'Predicted Damage Index (0–1 Scale)'
X = df_final.copy()
y = df[main_target].copy()

print(f"X (features): {X.shape}")
print(f"y (target): {y.shape}")
print(f"Target principal: {main_target}")

In [ ]:
# División 80/20 con reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("=== División de Datos ===")
print(f"Conjunto de Entrenamiento: {X_train.shape[0]} filas (80%)")
print(f"Conjunto de Prueba:        {X_test.shape[0]} filas (20%)")
print(f"\nDistribución de la variable target:")
print(f"  Train - Media: {y_train.mean():.4f}, Std: {y_train.std():.4f}")
print(f"  Test  - Media: {y_test.mean():.4f}, Std: {y_test.std():.4f}")

In [ ]:
# Verificación de la división
print("=== Verificación del Split ===")
total = len(X)
train_pct = len(X_train) / total * 100
test_pct = len(X_test) / total * 100
print(f"Train: {len(X_train)} ({train_pct:.1f}%)")
print(f"Test:  {len(X_test)} ({test_pct:.1f}%)")
print(f"Total: {total}")
print(f"\nColumnas en train: {X_train.shape[1]}")
print(f"Columnas en test:  {X_test.shape[1]}")

In [ ]:
# División con todos los targets (multi-output)
targets_all = [t for t in df.columns if t.startswith('Predicted')]
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X, df[targets_all],
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f"Multi-output - Targets: {len(targets_all)}")
print(f"Train: {X_train_all.shape}, y_train_all: {y_train_all.shape}")
print(f"Test:  {X_test_all.shape}, y_test_all: {y_test_all.shape}")

### 3.3 Variable Objetivo y Dataset Final Codificado/Escalado
Definición de `Predicted Damage Index (0–1 Scale)` como variable objetivo (target) e impresión del dataset con características categóricas codificadas y numéricas escaladas junto con la variable objetivo.

In [ ]:
# Importar función de división
from sklearn.model_selection import train_test_split

# Variable objetivo principal
target_col = 'Predicted Damage Index (0–1 Scale)'
y = df[target_col].copy()

# Features: dataset final (numéricas escaladas + categóricas codificadas)
X = df_final.copy()

print(f"Variable objetivo: {target_col}")
print(f"X (features): {X.shape}")
print(f"y (target):   {y.shape}")

In [ ]:
# División 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("=== Conjuntos de Entrenamiento y Prueba ===")
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

In [ ]:
# Imprimir dataset final codificado y escalado junto con la variable objetivo
print("=== Dataset Final: Features Codificadas/Escaladas + Variable Objetivo ===")
print(f"Shape total: {X.shape}")
print(f"\nColumnas de features ({len(X.columns)}):")
for i, col in enumerate(X.columns):
    print(f"  {i+1}. {col}")

print(f"\nVariable objetivo (target): {target_col}")
print(f"Estadísticas del target:\n{y.describe()}")

In [ ]:
# Concatenar X y y para inspección visual
df_final_with_target = pd.concat([X, y], axis=1)

print(f"Dataset completo con target: {df_final_with_target.shape}")
display(df_final_with_target.head(10))

In [ ]:
# Resumen de features numéricas vs escaladas
print("=== Comparación: Numeric Original vs Escala ===")
for col in feature_cols:
    print(f"\n{col}:")
    print(f"  Original - Min: {df[col].min():.4f}, Max: {df[col].max():.4f}, Mean: {df[col].mean():.4f}")
    print(f"  Escalada - Min: {X[col].min():.4f}, Max: {X[col].max():.4f}, Mean: {X[col].mean():.4f}, Std: {X[col].std():.4f}")

In [ ]:
# Resumen de variables categóricas codificadas
encoded_cols = [c for c in X.columns if c not in feature_cols]
print(f"=== Variables Categóricas Codificadas (One-Hot) ===")
print(f"Total columnas codificadas: {len(encoded_cols)}")
for col in encoded_cols:
    print(f"  - {col}: {X[col].unique().tolist()}")

## Paso 4:  Entrenamiento y Evaluación de Modelos de Regresión

### 4.1 Importar Librerías y Definir Modelos
Implementación de modelos de regresión: Regresión Lineal, Random Forest, XGBoost y SVR. Se utilizarán las métricas MAE, RMSE y R² para evaluar el rendimiento predictivo.

In [ ]:
# Importar librerías de modelos y métricas
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Definir los modelos a comparar
models = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0, eval_metric='rmse'),
    'SVR': SVR(kernel='rbf', C=1.0)
}

print("=== Modelos Definidos ===")
for name, model in models.items():
    print(f"  - {name}: {model}")

In [ ]:
# Entrenar todos los modelos
trained_models = {}
for name, model in models.items():
    print(f"Entrenando {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"  -> Completado")

print("\n=== Todos los modelos han sido entrenados ===")

In [ ]:
# Evaluar cada modelo con MAE, RMSE y R2
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        'Modelo': name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print("=== Métricas de Evaluación (Test Set) ===")
display(results_df.to_string(index=False))

In [ ]:
# Tabla de resultados con formato y estilo
styled_results = results_df.style.format({
    'MAE': '{:.4f}',
    'RMSE': '{:.4f}',
    'R²': '{:.4f}'
}).background_gradient(cmap='Blues', subset=['R²'], axis=0).background_gradient(cmap='Reds', subset=['MAE', 'RMSE'], axis=0, ascending=True)
styled_results

In [ ]:
# Visualización comparativa de métricas
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE
axes[0].barh(results_df['Modelo'], results_df['MAE'], color='steelblue')
axes[0].set_title('MAE', fontweight='bold')
axes[0].set_xlabel('Error Absoluto Medio')

# RMSE
axes[1].barh(results_df['Modelo'], results_df['RMSE'], color='coral')
axes[1].set_title('RMSE', fontweight='bold')
axes[1].set_xlabel('Raíz del Error Cuadrático Medio')

# R2
axes[2].barh(results_df['Modelo'], results_df['R²'], color='seagreen')
axes[2].set_title('R²', fontweight='bold')
axes[2].set_xlabel('Coeficiente de Determinación')

for ax in axes:
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Comparación de Métricas por Modelo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Identificar el mejor modelo según R2
best_model_name = results_df.loc[results_df['R²'].idxmax(), 'Modelo']
best_r2 = results_df['R²'].max()
best_mae = results_df.loc[results_df['R²'].idxmax(), 'MAE']
best_rmse = results_df.loc[results_df['R²'].idxmax(), 'RMSE']

print(f"=== Mejor Modelo ===")
print(f"Modelo: {best_model_name}")
print(f"R²:     {best_r2:.4f}")
print(f"MAE:    {best_mae:.4f}")
print(f"RMSE:   {best_rmse:.4f}")

### 4.2 Análisis de Relación: Stories, Altura y Frecuencia Natural
Figura con dos subplots de dispersión para visualizar la relación entre el número de pisos, la altura del edificio y la frecuencia natural.

In [ ]:
# Figura con dos subplots de dispersión
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Number of Stories vs Natural Frequency (Hz)
axes[0].scatter(df['Number of Stories'], df['Natural Frequency (Hz)'],
                   alpha=0.6, c='steelblue', edgecolors='navy', s=50)
axes[0].set_xlabel('Number of Stories', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Natural Frequency (Hz)', fontsize=12, fontweight='bold')
axes[0].set_title('Number of Stories vs Natural Frequency', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Subplot 2: Building Height (m) vs Natural Frequency (Hz)
axes[1].scatter(df['Building Height (m)'], df['Natural Frequency (Hz)'],
                   alpha=0.6, c='coral', edgecolors='darkred', s=50)
axes[1].set_xlabel('Building Height (m)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Natural Frequency (Hz)', fontsize=12, fontweight='bold')
axes[1].set_title('Building Height vs Natural Frequency', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Relación entre Características de Construcción y Frecuencia Natural',
              fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### Análisis de Correlación entre Stories, Altura y Frecuencia Natural
Cálculo de las correlaciones para cuantificar la fuerza de las relaciones lineales observadas en los scatter plots.

In [145]:
# Correlación entre Number of Stories y Natural Frequency
corr_stories = df['Number of Stories'].corr(df['Natural Frequency (Hz)'])
# Correlación entre Building Height y Natural Frequency
corr_height = df['Building Height (m)'].corr(df['Natural Frequency (Hz)'])

print("=== Correlaciones con Natural Frequency (Hz) ===")
print(f"Number of Stories  vs Natural Frequency: {corr_stories:.4f}")
print(f"Building Height (m) vs Natural Frequency: {corr_height:.4f}")

if abs(corr_stories) > abs(corr_height):
    print("\n-> Number of Stories tiene una relación más fuerte con la frecuencia natural.")
else:
    print("\n-> Building Height tiene una relación más fuerte con la frecuencia natural.")

=== Correlaciones con Natural Frequency (Hz) ===
Number of Stories  vs Natural Frequency: -0.0049
Building Height (m) vs Natural Frequency: -0.0072

-> Building Height tiene una relación más fuerte con la frecuencia natural.
